In [1]:
from datasets import load_dataset

In [2]:
# loading MNLI dataset from GLUE
# 0 == entailment
# 1 == neutral
# 2 == contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns(["idx"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

In [3]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

## Train Model

In [4]:
from sentence_transformers import SentenceTransformer

# use a base model
embedding_model = SentenceTransformer("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
from sentence_transformers import losses

# define the loss function.
# in softmax loss- we will also need to explicitly set the number of layers
train_loss = losses.SoftmaxLoss(
    model = embedding_model,
    sentence_embedding_dimension = embedding_model.get_sentence_embedding_dimension(),
    num_labels = 3
)

In [6]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# create an embedding similarity evaluator for STSB
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [7]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "base_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

In [8]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# train embedding model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: thesage210 (thesage210-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.077400
200,0.951500
300,0.875200
400,0.833200
500,0.822000
600,0.818300
700,0.802400
800,0.776700
900,0.771600
1000,0.761200


TrainOutput(global_step=1563, training_loss=0.8060669584756316, metrics={'train_runtime': 409.3345, 'train_samples_per_second': 122.149, 'train_steps_per_second': 3.818, 'total_flos': 0.0, 'train_loss': 0.8060669584756316, 'epoch': 1.0})

In [9]:
# evaluate the model
evaluator(embedding_model)

{'pearson_cosine': 0.6121938162328866, 'spearman_cosine': 0.6703274928492675}

## Using Cosine Similarity for loss function



In [10]:
from datasets import Dataset

# neutral/contradiction = 0 and entailment = 1
mapping = {2:0, 1:0, 0:1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [11]:
# create same evaluator as before
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# create an embedding similarity evaluator for STSB
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

In [12]:
# then follow the same steps as before but select a different loss instead
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers import SentenceTransformer

# use a base model
embedding_model = SentenceTransformer("bert-base-uncased")

from sentence_transformers import losses

# define the loss function.
train_loss = losses.CosineSimilarityLoss(
    model = embedding_model
)

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

# train embedding model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.229600
200,0.171600
300,0.172200
400,0.157900
500,0.152700
600,0.157000
700,0.149200
800,0.156900
900,0.150300
1000,0.146500


TrainOutput(global_step=1563, training_loss=0.15707940561071237, metrics={'train_runtime': 406.6676, 'train_samples_per_second': 122.951, 'train_steps_per_second': 3.843, 'total_flos': 0.0, 'train_loss': 0.15707940561071237, 'epoch': 1.0})

In [14]:
evaluator(embedding_model)

{'pearson_cosine': 0.7274366052371852, 'spearman_cosine': 0.7290451988514102}

## Multiple Negatives Ranking Loss

In [20]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# load mnli dataset from glue
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))
mnli = mnli.remove_columns("idx")

mnli =  mnli.filter(lambda x: True if x["label"] == 0 else False)

# prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = mnli["hypothesis"]
soft_negatives = [row for row in soft_negatives]
random.shuffle(soft_negatives)
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)

train_dataset = Dataset.from_dict(train_dataset)

16875it [00:01, 14282.90it/s]


In [25]:
# definging the evaluator
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

In [26]:
# we train as before but with MNR loss instead
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers import SentenceTransformer

# define model
embedding_model = SentenceTransformer("bert-base-uncased")

from sentence_transformers import losses

# define the loss function.
train_loss = losses.MultipleNegativesRankingLoss(
    model = embedding_model)

# defining the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

In [27]:
# train model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.308500
200,0.101500
300,0.078900
400,0.065100
500,0.065900


TrainOutput(global_step=528, training_loss=0.12074665070483179, metrics={'train_runtime': 151.9088, 'train_samples_per_second': 111.086, 'train_steps_per_second': 3.476, 'total_flos': 0.0, 'train_loss': 0.12074665070483179, 'epoch': 1.0})

In [28]:
evaluator(embedding_model)

{'pearson_cosine': 0.8103647508940532, 'spearman_cosine': 0.8119599927265987}

## FINE TUNING AN EMBEDDING MODEL

### SUPERVISED

In [1]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

In [2]:
# load mnli dataset from GLUE
# entailment,neutral and contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns(["idx"])

README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

In [3]:
# create an embedding similarity evaluator for stsb
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [7]:
# train steps are similar as before but instead of using bert base uncased - use pretrained embedding model
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers import SentenceTransformer, losses

# use a base model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# loss function - same as MNR
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# train model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: thesage210 (thesage210-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,0.158300
200,0.113100
300,0.122400
400,0.119600
500,0.110400
600,0.101700
700,0.121300
800,0.101700
900,0.102500
1000,0.104300


TrainOutput(global_step=1563, training_loss=0.11035517401521357, metrics={'train_runtime': 139.8381, 'train_samples_per_second': 357.556, 'train_steps_per_second': 11.177, 'total_flos': 0.0, 'train_loss': 0.11035517401521357, 'epoch': 1.0})

In [9]:
evaluator(embedding_model)

{'pearson_cosine': 0.8492770770331721, 'spearman_cosine': 0.8491079329535359}

#### AUGMENTED SBERT

In [10]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader

In [14]:
# prepare a small set of 10000 documents for the cross encoder
dataset = load_dataset("glue", "mnli", split="train")
dataset = dataset.select(range(10_000))
mapping = {2:0, 1:0, 0:1}

# data loader
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

100%|██████████| 10000/10000 [00:00<00:00, 23255.31it/s]


In [15]:
# pandas dataframe for easier data handling -- gold dataset
gold = pd.DataFrame(
    {
        "sentence1":dataset["premise"],
        "sentence2":dataset["hypothesis"],
        "label":[mapping[label] for label in dataset["label"]]
    }
)

In [17]:
gold

,sentence1,sentence2,label
0,Conceptually cream skimming has two basic dime...,Product and geography are what make cream skim...,0
1,you know during the season and i guess at at y...,You lose the things to the following level if ...,1
2,One of our number will carry out your instruct...,A member of my team will execute your orders w...,1
3,How do you know? All this is their information...,This information belongs to them.,1
4,yeah i tell you what though if you go price so...,The tennis shoes have a range of prices.,0
...,...,...,...
9995,"Because, despite its monopoly power, Microsoft...",Microsoft owns 60 percent of all computer-rela...,0
9996,"'Right,' I mumbled.","'Wrong', I said.",0
9997,Thanks dad.,Thanks Obama.,0
9998,which is good,I don't think that's great,0


In [18]:
# step 1 - use gold dataset and train the cross encoder
from sentence_transformers.cross_encoder import CrossEncoder

cross_encoder = CrossEncoder("bert-base-uncased", num_labels = 2)
cross_encoder.fit(
    train_dataloader = gold_dataloader,
    epochs = 1,
    warmup_steps = 100,
    show_progress_bar=True,
    use_amp = False
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


In [19]:
# step -2 -- use the remaining 40000 sentence pairs as our silver dataset

# prepare the silver dataset by predicting labels with the cross-encoder
silver = load_dataset(
    "glue", "mnli", split="train"
).select(range(10_000, 50_000))
silver = silver.remove_columns(["idx"])
pairs = list(zip(silver["premise"], silver["hypothesis"]))

In [20]:
# step -3-- since these sentence pairs are unlabeled. we will use our fine-tuned encoder to label these sentence pairs

import numpy as np

# label the sentence pairs using our fine tuned cross- encoder
output = cross_encoder.predict(
    pairs,
    apply_softmax = True,
    show_progress_bar = True
)

silver = pd.DataFrame(
    {
        "sentence1":silver["premise"],
        "sentence2":silver["hypothesis"],
        "label":np.argmax(output, axis=1)
    }
)

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

In [21]:
# now we have  our gold and silver dataset , we simply combine them and train our embedding model as before

# combine gold and silver datasets
data = pd.concat([gold, silver], ignore_index = True, axis = 0)
data = data.drop_duplicates(subset = ["sentence1", "sentence2"], keep = "first")
train_dataset = Dataset.from_pandas(data, preserve_index = False)

In [22]:
train_dataset

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 49998
})

In [23]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# create an embedding similarity evaluator for stsb
val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

In [24]:
# we train the model same as before except now we use the augmented dataset
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers import SentenceTransformer, losses

# define model
embedding_model = SentenceTransformer("bert-base-uncased")

# loss function
train_loss = losses.CosineSimilarityLoss(model =embedding_model)

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

In [25]:
# train model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.216800
200,0.155600
300,0.142300
400,0.141500
500,0.139000
600,0.135200
700,0.133500
800,0.132000
900,0.129400
1000,0.130600


TrainOutput(global_step=1563, training_loss=0.13928585333162138, metrics={'train_runtime': 412.2014, 'train_samples_per_second': 121.295, 'train_steps_per_second': 3.792, 'total_flos': 0.0, 'train_loss': 0.13928585333162138, 'epoch': 1.0})

In [26]:
evaluator(embedding_model)

{'pearson_cosine': 0.7042538074421858, 'spearman_cosine': 0.71467715334868}

### UNSUPERVISED

#### TRANSFORMER-BASED SEQUENTIAL DENOISING AUTO-ENCODER ( TSDAE )

In [1]:
# we start by downloading an external tokenizer - which is used for the denoising procedure

# download additional tokenizer
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# create flat sentences from our data and remove any label that we have to mimic an unsupervised settting
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# create a flat list of sentences
mnli = load_dataset("glue", "mnli", split="train").select(range(25_000))
flat_sentences = [row["premise"] for row in tqdm(mnli)]
flat_sentences += [row["hypothesis"] for row in tqdm(mnli)]

# add noise to our input data
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))

# create dataset
train_dataset = {"damaged_sentence":[], "original_sentence":[]}

for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])

train_dataset = Dataset.from_dict(train_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

100%|██████████| 48353/48353 [00:07<00:00, 6389.60it/s]


In [3]:
train_dataset[0]

{'damaged_sentence': 'are given setting, of.',
 'original_sentence': 'Because simplicity and ease of administration are usually given some weight in rate setting, the number of worksharing discounts is limited.'}

In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# create an embedding similarity evaluator for stsb
val_sts = load_dataset("glue", "stsb", split= "validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity = "cosine"
)

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [5]:
# CLS token as the pooling strategy instead of mean pooling of the token embeddings

from sentence_transformers import models, SentenceTransformer

# create embedding model
word_embedding_model = models.Transformer("bert-base-uncased")
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), "cls")
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
from sentence_transformers import losses

# use the denoising auto-encoder loss
train_loss = losses.DenoisingAutoEncoderLoss(
    model = embedding_model,
    tie_encoder_decoder = True)

train_loss.decoder = train_loss.decoder.to("cuda")

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

In [7]:
# training
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir = "tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps = 100,
    fp16 = True,
    eval_steps = 100,
    logging_steps = 100
)

In [8]:
# train model
trainer = SentenceTransformerTrainer(
    model = embedding_model,
    args = args,
    train_dataset = train_dataset,
    loss = train_loss,
    evaluator = evaluator
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: thesage210 (thesage210-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Step,Training Loss
100,6.854300
200,4.918100
300,4.623300
400,4.519400
500,4.364300
600,4.285800
700,4.167800
800,4.146700
900,4.068900
1000,4.037100


TrainOutput(global_step=3023, training_loss=4.020929887363384, metrics={'train_runtime': 1268.2877, 'train_samples_per_second': 38.125, 'train_steps_per_second': 2.384, 'total_flos': 0.0, 'train_loss': 4.020929887363384, 'epoch': 1.0})

In [9]:
evaluator(embedding_model)

{'pearson_cosine': 0.7326031263004785, 'spearman_cosine': 0.7389768540490521}